# Phase Envelope Calculation with NeqSim

This notebook demonstrates how to calculate PT phase envelopes using NeqSim, including:

- Basic phase envelope with dew point, bubble point, cricondenbar, and cricondentherm
- Envelope closure verification
- **Quality lines** (constant molar vapor fraction curves) with volume and mass fractions
- Plotting the complete phase diagram
- Effect of composition on the phase envelope

## 1. Setup

In [ ]:
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except ImportError:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import NeqSim classes
if NEQSIM_MODE == "devtools":
    pass  # Classes already on ns.*
else:
    ns = type('ns', (), {})()
    ns.SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ns.SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ns.ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
    # PTPhaseEnvelope for direct access to quality line methods
    ns.PTPhaseEnvelope = jneqsim.thermodynamicoperations.phaseenvelopeops.multicomponentenvelopeops.PTPhaseEnvelope

## 2. Basic Phase Envelope

Create a natural gas fluid and calculate the phase envelope.

In [ ]:
# Create a natural gas fluid
# Temperature in Kelvin, pressure in bara
fluid = ns.SystemSrkEos(273.15 + 20.0, 50.0)
fluid.addComponent("nitrogen", 0.88)
fluid.addComponent("CO2", 5.70)
fluid.addComponent("methane", 86.89)
fluid.addComponent("ethane", 3.59)
fluid.addComponent("propane", 1.25)
fluid.addComponent("i-butane", 0.19)
fluid.addComponent("n-butane", 0.35)
fluid.addComponent("i-pentane", 0.12)
fluid.addComponent("n-pentane", 0.12)
fluid.setMixingRule("classic")  # MANDATORY: always set a mixing rule

# Calculate phase envelope
ops = ns.ThermodynamicOperations(fluid)
ops.calcPTphaseEnvelope()

# Extract envelope data
dewT = np.array(list(ops.get("dewT"))) - 273.15   # Convert K to °C
dewP = np.array(list(ops.get("dewP")))
bubT = np.array(list(ops.get("bubT"))) - 273.15
bubP = np.array(list(ops.get("bubP")))

# Critical points
cricondenbar = list(ops.get("cricondenbar"))
cricondentherm = list(ops.get("cricondentherm"))
critical = list(ops.get("criticalPoint1"))

print(f"Cricondenbar:   {cricondenbar[1]:.2f} bara at {cricondenbar[0] - 273.15:.1f} °C")
print(f"Cricondentherm: {cricondentherm[0] - 273.15:.1f} °C at {cricondentherm[1]:.2f} bara")
print(f"Critical point: {critical[0] - 273.15:.1f} °C, {critical[1]:.2f} bara")
print(f"Dew curve:    {len(dewT)} points")
print(f"Bubble curve: {len(bubT)} points")

# Check envelope closure
envelope = ops.getOperation()
print(f"Envelope closed: {envelope.isEnvelopeClosed()}")

## 3. Plot the Phase Envelope

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Plot dew and bubble curves
ax.plot(dewT, dewP, 'b-', linewidth=2, label='Dew Point Curve')
ax.plot(bubT, bubP, 'r-', linewidth=2, label='Bubble Point Curve')

# Mark special points
ax.plot(cricondenbar[0] - 273.15, cricondenbar[1], 'ko', markersize=10,
        label=f'Cricondenbar ({cricondenbar[1]:.1f} bara)')
ax.plot(cricondentherm[0] - 273.15, cricondentherm[1], 'g^', markersize=10,
        label=f'Cricondentherm ({cricondentherm[0] - 273.15:.1f} °C)')
ax.plot(critical[0] - 273.15, critical[1], 'rs', markersize=10,
        label=f'Critical Point ({critical[0] - 273.15:.1f} °C, {critical[1]:.1f} bara)')

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (bara)', fontsize=12)
ax.set_title('PT Phase Envelope — Natural Gas', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-200, max(dewT.max(), bubT.max()) + 20)
plt.tight_layout()
plt.savefig('phase_envelope_basic.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Phase Envelope with Quality Lines

Quality lines are curves of constant molar vapor fraction inside the two-phase region.
At each point on a quality line the volume fraction and mass fraction are also computed.

The `calcPTphaseEnvelopeWithQualityLines` method first traces the full phase envelope,
then traces quality lines at the specified molar vapor fractions.

In [ ]:
# Create the same fluid
fluid2 = ns.SystemSrkEos(273.15 + 20.0, 50.0)
fluid2.addComponent("nitrogen", 0.88)
fluid2.addComponent("CO2", 5.70)
fluid2.addComponent("methane", 86.89)
fluid2.addComponent("ethane", 3.59)
fluid2.addComponent("propane", 1.25)
fluid2.addComponent("i-butane", 0.19)
fluid2.addComponent("n-butane", 0.35)
fluid2.addComponent("i-pentane", 0.12)
fluid2.addComponent("n-pentane", 0.12)
fluid2.setMixingRule("classic")

ops2 = ns.ThermodynamicOperations(fluid2)

# Calculate envelope + quality lines in one call
qualities = [0.1, 0.25, 0.5, 0.75, 0.9]
ops2.calcPTphaseEnvelopeWithQualityLines(qualities)

# Extract envelope data
dewT2 = np.array(list(ops2.get("dewT"))) - 273.15
dewP2 = np.array(list(ops2.get("dewP")))
bubT2 = np.array(list(ops2.get("bubT"))) - 273.15
bubP2 = np.array(list(ops2.get("bubP")))

# Print quality line summary
env2 = ops2.getOperation()
traced_q = list(env2.getQualityValues())
print(f"Traced {len(traced_q)} quality lines: {traced_q}")

for q in traced_q:
    data = env2.getQualityLine(q)
    if data is not None:
        npts = len(list(data[0]))
        # Volume fraction differs from molar fraction
        vol_fracs = list(data[2])
        mass_fracs = list(data[3])
        print(f"  beta={q:.2f}: {npts} points, "
              f"vol_frac=[{vol_fracs[0]:.3f}..{vol_fracs[-1]:.3f}], "
              f"mass_frac=[{mass_fracs[0]:.3f}..{mass_fracs[-1]:.3f}]")

In [ ]:
# Plot envelope with quality lines
fig, ax = plt.subplots(figsize=(10, 7))

# Outer envelope
ax.plot(dewT2, dewP2, 'b-', linewidth=2.5, label='Dew Point')
ax.plot(bubT2, bubP2, 'r-', linewidth=2.5, label='Bubble Point')

# Quality lines
cmap = plt.cm.viridis
for i, q in enumerate(traced_q):
    # Access via get() string keys
    qT = ops2.get(f"qualityT_{q}")
    qP = ops2.get(f"qualityP_{q}")
    if qT is not None:
        qT_arr = np.array(list(qT)) - 273.15
        qP_arr = np.array(list(qP))
        color = cmap(i / max(len(traced_q) - 1, 1))
        ax.plot(qT_arr, qP_arr, '--', color=color, linewidth=1.5,
                label=f'{q*100:.0f}% vapor (molar)')

# Critical points
cbar = list(ops2.get("cricondenbar"))
ctherm = list(ops2.get("cricondentherm"))
crit = list(ops2.get("criticalPoint1"))

ax.plot(cbar[0] - 273.15, cbar[1], 'ko', markersize=8)
ax.plot(ctherm[0] - 273.15, ctherm[1], 'g^', markersize=8)
ax.plot(crit[0] - 273.15, crit[1], 'rs', markersize=8)

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (bara)', fontsize=12)
ax.set_title('PT Phase Envelope with Quality Lines', fontsize=14)
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase_envelope_quality_lines.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Comparing Molar, Volume, and Mass Fractions

At the same molar vapor fraction, the volume and mass fractions differ because:
- **Volume fraction** > molar fraction (vapor is less dense, occupies more space)
- **Mass fraction** depends on phase molecular weight difference

The plot below shows how these three fractions differ at 50% molar vapor (beta = 0.5).

In [ ]:
# Compare volume and mass fractions for the 0.5 quality line
q_target = 0.5
data_50 = env2.getQualityLine(q_target)

if data_50 is not None:
    T_50 = np.array(list(data_50[0])) - 273.15
    P_50 = np.array(list(data_50[1]))
    vol_50 = np.array(list(data_50[2]))
    mass_50 = np.array(list(data_50[3]))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: fractions vs pressure
    ax1.axhline(y=q_target, color='gray', linestyle=':', linewidth=1,
                label=f'Molar fraction = {q_target}')
    ax1.plot(P_50, vol_50, 'b-', linewidth=2, label='Volume fraction')
    ax1.plot(P_50, mass_50, 'r-', linewidth=2, label='Mass fraction')
    ax1.set_xlabel('Pressure (bara)', fontsize=12)
    ax1.set_ylabel('Vapor Fraction (-)', fontsize=12)
    ax1.set_title('Vapor Fractions vs Pressure at 50% Molar Vapor', fontsize=12)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)

    # Right: fractions vs temperature
    ax2.axhline(y=q_target, color='gray', linestyle=':', linewidth=1,
                label=f'Molar fraction = {q_target}')
    ax2.plot(T_50, vol_50, 'b-', linewidth=2, label='Volume fraction')
    ax2.plot(T_50, mass_50, 'r-', linewidth=2, label='Mass fraction')
    ax2.set_xlabel('Temperature (°C)', fontsize=12)
    ax2.set_ylabel('Vapor Fraction (-)', fontsize=12)
    ax2.set_title('Vapor Fractions vs Temperature at 50% Molar Vapor', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('fraction_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"No quality line data available for beta = {q_target}")

## 6. Effect of Heavy Components on Phase Envelope

Adding heavier components shifts the phase envelope to higher temperatures and pressures.
This is why accurate characterization of C6+ fractions is critical for gas quality analysis.

In [ ]:
def create_gas(heavy_fraction=0.0):
    """Create a natural gas with optional heavy fraction."""
    f = ns.SystemSrkEos(273.15 + 20.0, 50.0)
    total_light = 0.90 - heavy_fraction
    f.addComponent("nitrogen", 0.01)
    f.addComponent("CO2", 0.02)
    f.addComponent("methane", total_light)
    f.addComponent("ethane", 0.04)
    f.addComponent("propane", 0.02)
    f.addComponent("n-butane", 0.01)
    if heavy_fraction > 0:
        f.addComponent("n-hexane", heavy_fraction)
    f.setMixingRule("classic")
    return f


fig, ax = plt.subplots(figsize=(10, 7))
fractions = [0.0, 0.005, 0.01, 0.02]
colors = ['blue', 'green', 'orange', 'red']

for frac, color in zip(fractions, colors):
    gas = create_gas(frac)
    ops_i = ns.ThermodynamicOperations(gas)
    ops_i.calcPTphaseEnvelope()

    dT = np.array(list(ops_i.get("dewT"))) - 273.15
    dP = np.array(list(ops_i.get("dewP")))
    bT = np.array(list(ops_i.get("bubT"))) - 273.15
    bP = np.array(list(ops_i.get("bubP")))

    ctherm = list(ops_i.get("cricondentherm"))

    label = f'No C6' if frac == 0 else f'{frac*100:.1f}% n-C6'
    label += f' (Tcrico={ctherm[0]-273.15:.0f}°C)'
    ax.plot(dT, dP, '-', color=color, linewidth=1.5, label=label)
    ax.plot(bT, bP, '-', color=color, linewidth=1.5)

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (bara)', fontsize=12)
ax.set_title('Effect of Heavy Components on Phase Envelope', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('composition_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Phase Envelope with Peng-Robinson EOS

Different equations of state give slightly different envelopes. The example below
compares SRK and PR for the same gas composition.

In [ ]:
# SRK
fluid_srk = ns.SystemSrkEos(273.15, 50.0)
fluid_srk.addComponent("methane", 0.85)
fluid_srk.addComponent("ethane", 0.08)
fluid_srk.addComponent("propane", 0.04)
fluid_srk.addComponent("n-butane", 0.03)
fluid_srk.setMixingRule("classic")

ops_srk = ns.ThermodynamicOperations(fluid_srk)
ops_srk.calcPTphaseEnvelope()

# PR
fluid_pr = ns.SystemPrEos(273.15, 50.0)
fluid_pr.addComponent("methane", 0.85)
fluid_pr.addComponent("ethane", 0.08)
fluid_pr.addComponent("propane", 0.04)
fluid_pr.addComponent("n-butane", 0.03)
fluid_pr.setMixingRule("classic")

ops_pr = ns.ThermodynamicOperations(fluid_pr)
ops_pr.calcPTphaseEnvelope()

fig, ax = plt.subplots(figsize=(10, 7))

for label, ops_eos, color in [("SRK", ops_srk, "blue"), ("PR", ops_pr, "red")]:
    dT = np.array(list(ops_eos.get("dewT"))) - 273.15
    dP = np.array(list(ops_eos.get("dewP")))
    bT = np.array(list(ops_eos.get("bubT"))) - 273.15
    bP = np.array(list(ops_eos.get("bubP")))
    crit_eos = list(ops_eos.get("criticalPoint1"))

    ax.plot(dT, dP, '-', color=color, linewidth=2, label=f'{label} Dew')
    ax.plot(bT, bP, '--', color=color, linewidth=2, label=f'{label} Bubble')
    ax.plot(crit_eos[0] - 273.15, crit_eos[1], 'o', color=color, markersize=8)

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (bara)', fontsize=12)
ax.set_title('Phase Envelope: SRK vs Peng-Robinson', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('eos_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary of API Methods

| Python Method | Description |
|---|---|
| `ops.calcPTphaseEnvelope()` | Default envelope (dew first, lowPres=1 bar) |
| `ops.calcPTphaseEnvelope(True)` | Start from bubble point side |
| `ops.calcPTphaseEnvelope(True, 0.5)` | Bubble first, starting at 0.5 bara |
| `ops.calcPTphaseEnvelopeWithQualityLines([0.1, 0.5, 0.9])` | Envelope + quality lines |
| `ops.get("dewT")` | Dew point temperatures (K) |
| `ops.get("dewP")` | Dew point pressures (bara) |
| `ops.get("bubT")` | Bubble point temperatures (K) |
| `ops.get("bubP")` | Bubble point pressures (bara) |
| `ops.get("cricondenbar")` | [T(K), P(bara)] at maximum pressure |
| `ops.get("cricondentherm")` | [T(K), P(bara)] at maximum temperature |
| `ops.get("criticalPoint1")` | [Tc(K), Pc(bara)] critical point |
| `ops.get("qualityT_0.5")` | Temperatures (K) at 50% quality |
| `ops.get("qualityP_0.5")` | Pressures (bara) at 50% quality |
| `ops.get("qualityVolFrac_0.5")` | Volume fractions at 50% quality |
| `ops.get("qualityMassFrac_0.5")` | Mass fractions at 50% quality |
| `envelope.isEnvelopeClosed()` | True if both branches were traced |
| `envelope.getQualityValues()` | Array of traced quality values |
| `envelope.getQualityLine(0.5)` | `[T[], P[], volFrac[], massFrac[]]` |

## Further Reading

- **Algorithm details**: See [Phase Envelope Algorithm](../docs/thermodynamicoperations/phase_envelope_algorithm.md) for the full mathematical formulation (Michelsen continuation method, Jacobian, critical point detection, step size control)
- **Usage guide**: See [Phase Envelope and Critical Points Guide](../docs/pvtsimulation/phase_envelope_guide.md) for Java examples and troubleshooting
- **Reference**: Michelsen, M.L. & Mollerup, J.M., *Thermodynamic Models: Fundamentals & Computational Aspects*, 2nd ed., 2007